# 第 13 章の実習　分析手順と再現性

2012 年の 6 月と 12 月について，登録または未登録の一方を選び，同じ関数で時刻別の平均貸出件数を比べます。関数が受け取る入力と，返された表を次の処理で使うことを確かめます。

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

1 行は一つの日付の一つの時計時刻帯の集計です。`cnt` は全体の貸出件数，`registered` と `casual` は登録・未登録の区分の件数です。同じ人が何度利用しても貸出の記録として数えるため，利用者の人数ではありません。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

file_name = "bike-sharing-hourly.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

hourly = pd.read_csv(csv_path)
hourly["date"] = pd.to_datetime(hourly["dteday"])
june = hourly.loc[
    (hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 6)
]
december = hourly.loc[
    (hourly["date"].dt.year == 2012) & (hourly["date"].dt.month == 12)
]
print("Python:", sys.version.split()[0], "pandas:", pd.__version__)
print("June rows:", len(june), "December rows:", len(december))
print("Periods:", june["dteday"].min(), "to", june["dteday"].max(), "/",
      december["dteday"].min(), "to", december["dteday"].max())
june[["dteday", "hr", "registered", "casual", "cnt"]].head(3)

## 1. 比べる区分を選ぶ（約 5 分）

本文では，全体の平均を比べました。2012 年の 6 月では 8 時が 508.4 件，17 時が 680.7 件でした。12 月では約 359.81 件と 375.97 件で，朝夕の高さが近づいていました。

今回は `registered`（登録）か `casual`（未登録）の一方を，結果を見る前に選びます。知りたいことと選んだ理由を一文で書いてください。どちらの区分も，利用目的を直接表す列ではありません。

- 選ぶ区分と理由：
- 6 月と 12 月で朝・昼・夕方のどの違いを調べたいか：

次の二つのセルで，**合計二行**を完成します。関数は入力表を `hr` で集計済みです。まず `return None` を，集計表を返す行へ書き換えます。次に `selected_column = None` を，選んだ列名の文字列へ書き換えます。未記入のままでは後続セルに課題未完と表示されます。

In [ ]:
def hourly_summary(records, value_column):
    grouped = records.groupby("hr")[value_column]
    summary = grouped.agg(recorded_hours="size", total="sum", mean="mean")
    return None  # 課題：集計表 summary を返す行に替える。

In [ ]:
selected_column = None  # 課題："registered" または "casual" を記入する。

## 2. 同じ関数へ二つの表を渡す（約 9 分）

次のセルは，選んだ区分について `hourly_summary(june, selected_column)` と `hourly_summary(december, selected_column)` を呼びます。入力の表は異なり，集計規則は同じです。返された表の 3・4・8・13・17 時を並べます。

実行前に，呼出の `june` と `selected_column` がどのパラメーターへ渡されるかを確認してください。表示を読むときは，`recorded_hours` が 6 月では 30，12 月の一部では 30，ほかでは 31 になることも確かめます。

In [ ]:
june_profile = None
december_profile = None
if selected_column not in ["registered", "casual"]:
    print("課題未完：selected_column に選んだ列名を文字列で指定してください。")
else:
    june_profile = hourly_summary(june, selected_column)
    december_profile = hourly_summary(december, selected_column)
    if june_profile is None or december_profile is None:
        print("課題未完：関数の return で集計表を返してください。")
    else:
        comparison = pd.concat(
            {"June": june_profile, "December": december_profile}, names=["period"]
        )
        display(comparison.loc[(slice(None), [3, 4, 8, 13, 17]), :].round(2))

In [ ]:
if june_profile is None or december_profile is None:
    print("課題未完：返り値の表ができたら，このセルを再実行してください。")
else:
    titles = {"registered": "Registered rentals", "casual": "Casual rentals"}
    fig, ax = plt.subplots(layout="constrained", figsize=(7, 4))
    ax.plot(june_profile.index, june_profile["mean"], "o-", markersize=3, label="June 2012")
    ax.plot(december_profile.index, december_profile["mean"], "s--", markersize=3, label="December 2012")
    upper = max(june_profile["mean"].max(), december_profile["mean"].max()) * 1.1
    ax.set(xlabel="Clock hour", ylabel="Mean rental count", title=titles[selected_column],
           xticks=range(0, 24, 3), ylim=(0, upper))
    ax.legend(frameon=False)
    plt.show()
    print("Highest mean in June:", june_profile["mean"].idxmax(), "h")
    print("Highest mean in December:", december_profile["mean"].idxmax(), "h")

図で一つの山を選び，返り値の行へ戻って，時刻・合計・行数・平均を記録します。6 月と 12 月で高さや位置がどう違うかを一文で説明してください。

12 月の 3・4 時は各 30 行から平均しています。元の表でその時刻がない日を 0 件として追加したものではありません。報告では「収録された時間帯の平均」と分かるようにします。

- 図の一点を作る合計・分母・平均：
- 6 月と 12 月の違い：
- この図だけではまだ分からないこと：

## 3. 再実行して照合する（約 6 分）

ここまでのセルを保存し，カーネル（Colab ではランタイム）の再起動後に，上から全セルを実行します。二行の記入内容はノートに保存されますが，実行時に作った変数と関数は作り直されます。

次のセルで，時刻別に分けた合計を足すと元の全件数に戻るか，行数を失っていないか，8 時の平均が元の行から直接求めた平均と一致するかを確かめます。同じ数字が再び出ることと，その数字が何を表すかの照合を分けて考えます。

In [ ]:
if june_profile is None or december_profile is None:
    print("課題未完：二つの返り値ができてから照合します。")
else:
    checks = []
    for label, records, profile in [
        ("June", june, june_profile), ("December", december, december_profile)
    ]:
        eight = records.loc[records["hr"] == 8, selected_column]
        checks.append({
            "period": label,
            "rows_match": profile["recorded_hours"].sum() == len(records),
            "totals_match": profile["total"].sum() == records[selected_column].sum(),
            "eight_matches": profile.loc[8, "mean"] == eight.mean(),
        })
    display(pd.DataFrame(checks))

次の項目が伝わるよう，図と短表に添える 3〜4 文の報告を書きます。

- 入力データの版，期間，選んだ区分。
- 関数が共通にした規則と，返り値で読んだ時刻・件数の違い。
- 12 月の分母を含めた観測範囲と，利用目的や原因についてまだ言えないこと。
- 最初から再実行できたか，どの出力を元の記録と照合したか。

（ここに報告を記入）

今回の分析は乱数を使っていません。第 10 章の並べ替えのように乱数を使う分析を再現する場合には，追加で何を記録するか一つ挙げてください。

<details>
<summary>二行の記入例と結果の確認</summary>

関数の最後は `return summary`，選択は例えば `selected_column = "registered"` とします。登録区分の時刻別平均は，6 月では 17 時が最大で約 558.23 件，12 月では 8 時が最大で約 350.06 件です。朝夕の高さの関係が変わったことを，図と該当する行で説明できます。登録区分だから全件が通勤とは限りません。

`"casual"` を選んだ場合は，6 月の最大が 17 時の約 122.47 件，12 月の最大が 14 時の約 48.42 件です。答えの最大時刻だけでなく，近い時間帯の形と件数も図で読んでください。

どちらを選んでも12月の3・4時は各30行からの平均です。再実行と照合ができても，欠けた時刻の実際の利用が0だったと証明したことにはなりません。乱数を使う別の分析なら，シードや反復回数も再現条件として残します。

</details>

## 出典

Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository，CC BY 4.0。 [配布元](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) ・ [データ DOI](https://doi.org/10.24432/C5W894)

本書の `bike-sharing-hourly.csv` は UCI 公式 ZIP の `hour.csv` を改名した集計版です。列名・値・行順は同じです。現在の事業者の個票ログとは版が異なり，混ぜて使いません。日付・時計時刻・欠落時刻の性格は Book の出典説明と第 11 章を参照してください。